# Data, EDA and Modeling
This notebook demonstrates the data loading, basic EDA, cleaning steps, and training a simple model. It saves the best model and training column names to `models/`.

In [ ]:
# Imports
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import joblib
print('ready')

In [ ]:
# Load cleaned CSV if present, else fallback to extracted files
candidates = ['diabetes_cleaned.csv', 'diabetes_extracted_fixed.csv', 'diabetes_extracted.csv']
df = None
for c in candidates:
    if os.path.exists(c):
        df = pd.read_csv(c)
        print('Loaded', c, 'shape=', df.shape)
        break
if df is None:
    raise FileNotFoundError('No dataset found in repository root. Run extraction/preprocess scripts first.')

In [ ]:
# Simple EDA: value counts for target if present
if 'CLASS' in df.columns:
    print(df['CLASS'].value_counts(dropna=False))
else:
    print('No CLASS column found')

In [ ]:
# Basic cleaning: drop obvious ID columns if present
for col in ['ID','No_Pation','No_Pation ']:
    if col in df.columns:
        df = df.drop(columns=[col])
df.shape

In [ ]:
# Prepare labeled subset and features
# Map CLASS to 0/1 if needed
if 'CLASS' in df.columns:
    df['CLASS_mapped'] = df['CLASS'].map({'Y':1,'N':0,1:1,0:0}).astype('float')
    labeled = df[df['CLASS_mapped'].notna()].copy()
    print('Labeled rows:', labeled.shape[0])
else:
    raise RuntimeError('CLASS column required for modeling')

In [ ]:
# Select numeric columns for a simple baseline
numeric_cols = labeled.select_dtypes(include=[np.number]).columns.tolist()
# remove target and any columns that clearly leak label tokens if present
for bad in ['Y','N','P','CLASS_mapped','CLASS']:
    if bad in numeric_cols:
        numeric_cols.remove(bad)
print('Using numeric columns count=', len(numeric_cols))
X = labeled[numeric_cols].fillna(labeled[numeric_cols].mean())
y = labeled['CLASS_mapped'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
model = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
print('Train score', model.score(X_train_s, y_train))
print('Test score', model.score(X_test_s, y_test))
os.makedirs('models', exist_ok=True)
joblib.dump(model, os.path.join('models','diabetes_model.pkl'))
joblib.dump(numeric_cols, os.path.join('models','training_columns.pkl'))
joblib.dump(scaler, os.path.join('models','scaler.pkl'))
print('Saved model and artifacts to models/')